# Why churn prediction is the wrong question

A 10-minute walkthrough of PRISM's central argument, using the artifacts written by

```bash
python -m prism.pipelines.run_all --config configs/fast.yaml --steps all
```

Nothing is retrained here: this notebook reads `artifacts/reports/` and re-derives the
headline claims so they can be inspected rather than taken on trust.

**The argument, in one line:** the customers most likely to churn and the customers most
*worth contacting* are different people, and mistaking one for the other costs money.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
REPORTS = ROOT / "artifacts" / "reports"

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

if not (REPORTS / "metrics.json").exists():
    raise SystemExit(
        "No pipeline output found. Run this first:\n"
        "  python -m prism.pipelines.run_all --config configs/fast.yaml --steps all"
    )

metrics = json.loads((REPORTS / "metrics.json").read_text(encoding="utf-8"))
print(f"run: {metrics.get('model_version')}  |  estimator: {metrics.get('estimator')}")
print(f"panel rows: {metrics.get('n_rows_panel'):,}  |  customers: {metrics.get('n_customers'):,}")

## 1. Four kinds of customer, not two

A churn model sorts people by *whether they will leave*. What actually matters is how they
respond to being contacted. Radcliffe & Surry's taxonomy splits the base four ways, and only
one of those groups is worth spending on.

The table below is the mean effect of each offer on 12-month discounted margin, by segment.
These are the simulator's *known* counterfactuals, computed analytically.

In [ ]:
segments = pd.read_csv(REPORTS / "segment_effects.csv").set_index("gt_segment")
display(segments.round(2))

best = segments.max(axis=1)
print(f"\nPersuadables gain {best['persuadable']:+.1f} from their best offer.")
print(f"Sleeping dogs LOSE {abs(best['sleeping_dog']):.1f} from theirs -- contacting them destroys value.")
print("\nA churn model cannot distinguish these two groups. That is the whole problem.")

## 2. The naive answer is not just imprecise, it has the wrong sign

Treatment was not assigned at random in the observational log: customers who looked risky
were contacted more. So comparing treated to untreated averages measures the selection, not
the offer.

`confounding_bias.csv` puts the naive comparison next to an adjusted estimate and the truth.

In [ ]:
path = REPORTS / "confounding_bias.csv"
if path.exists():
    bias = pd.read_csv(path)
    cols = [c for c in ["arm_name", "true_ate", "naive_diff_in_means", "ipw_estimate",
                        "model_estimate", "naive_sign_wrong"] if c in bias.columns]
    display(bias[cols].round(2))
    if "naive_sign_wrong" in bias:
        n_wrong = int(bias["naive_sign_wrong"].sum())
        print(f"\nThe naive comparison gets the SIGN wrong on {n_wrong} of {len(bias)} offers.")
    if "bias_reduction_vs_naive_pct" in metrics:
        print(f"Adjustment removes {metrics['bias_reduction_vs_naive_pct']:.0f}% of the naive bias.")
else:
    print("confounding_bias.csv not found -- re-run the pipeline to generate it.")

## 3. Does the estimator actually recover the truth?

This is the question no real project can answer, because the individual treatment effect
`tau_i` is never observed. Here it *is* observed, because the simulator computed it, so the
estimators can be graded with **PEHE** (root-mean-square error of the estimated effect).

The honest baseline is a constant-effect model: one that knows the true average effect but no
heterogeneity at all. Beating it is the minimum bar for claiming the model learned anything.

In [ ]:
lb = pd.read_csv(REPORTS / "leaderboard.csv")
display(lb.round(4))

baseline = metrics.get("baseline_constant_pehe")
best_pehe = metrics.get("best_learner_pehe")
if baseline and best_pehe:
    print(f"\nbest PEHE {best_pehe:.3f}  vs  constant-effect baseline {baseline:.3f}")
    print(f"-> the model explains {100 * (1 - best_pehe / baseline):.0f}% more of the effect variation.")

## 4. Time-to-event, not a binary flag

PRISM's distinctive estimator targets **restricted mean survival time** rather than a churn
flag. An offer that delays churn by three months scores zero against a 12-month binary
outcome, yet three months of margin is real money.

`survival_curves.png` shows the counterfactual retention curve under each offer; the vertical
gap at each month is the retained share that offer buys.

In [ ]:
from IPython.display import Image, display as show

FIG = ROOT / "docs" / "figures"
for name in ("survival_curves.png", "uplift_deciles.png"):
    if (FIG / name).exists():
        show(Image(filename=str(FIG / name)))
    else:
        print(f"{name} not generated")

## 5. From an effect to a decision

An effect estimate is not a decision. Each offer has a cost, the budget is finite, and each
customer can receive at most one offer -- a *multiple-choice knapsack*.

Solving it by Lagrangian relaxation produces a shadow price `lambda*`, which is the marginal
return on the next unit of budget. That single number is what actually settles a budget
argument, and a greedy sort cannot produce it.

In [ ]:
policies = pd.read_csv(REPORTS / "policy_comparison.csv")
display(policies.round(2))

gain = metrics.get("policy_uplift_vs_risk_targeting")
risk = metrics.get("policy_value_highest_risk")
if gain is not None and risk:
    print(f"\nCausal targeting beats risk targeting by {gain:,.0f} ({100 * gain / abs(risk):+.0f}%).")
lam = metrics.get("policy_shadow_price")
if lam is not None:
    print(f"Shadow price at the chosen budget: {lam:.2f}"
          + ("  <- below 1.0: the last offers funded destroy value" if lam < 1 else ""))

In [ ]:
for name in ("policy_comparison.png", "efficient_frontier.png"):
    if (FIG / name).exists():
        show(Image(filename=str(FIG / name)))

## 6. How wrong could this be?

Every number above rests on **unconfoundedness**: that nothing we failed to measure drives
both the offer and the churn. In this dataset that assumption is *false by construction* --
the simulator generates a hidden confounder and withholds it from the feature set.

So the right question is not "is the assumption true" (it never is) but "how strong would the
violation have to be to overturn the conclusion?"

In [ ]:
path = REPORTS / "refutation.csv"
if path.exists():
    display(pd.read_csv(path).round(4))
else:
    print("refutation.csv not generated (set causal.refute: true)")

ev = metrics.get("e_value_point")
if ev:
    print(f"\nE-value: {ev:.2f}")
    print("An unmeasured confounder would need associations of at least this strength with")
    print("BOTH the offer and churn to explain the estimated effect away.")

captured = metrics.get("oracle_value_captured_pct")
if captured is not None:
    print(f"\nShare of the oracle's achievable value captured: {captured:.1f}%")

## What to take away

1. **Churn risk is the wrong ranking.** Sleeping dogs look risky and are harmed by contact;
   lost causes look risky and cannot be saved. Neither is worth budget.
2. **Naive comparisons can have the wrong sign**, not merely the wrong magnitude, when
   treatment was targeted rather than randomised.
3. **Delay is value.** Modelling time-to-event rather than a binary flag captures the offers
   that postpone churn instead of preventing it.
4. **The decision needs an optimiser, not a threshold** -- and the shadow price is the number
   the budget conversation actually turns on.
5. **Say how wrong it could be.** The refutation suite and E-value are the difference between
   an estimate and a claim.

Method details: [`docs/METHODOLOGY.md`](../docs/METHODOLOGY.md).
Design tradeoffs and rejected alternatives: [`docs/DECISIONS.md`](../docs/DECISIONS.md).